#FAS2rDNA Colab

In [ ]:
import os
from google.colab import files

# @title ##1. Upload data
# @markdown ###### Feed your files to FAS2rDNA: click 'Runtime' --> select 'Run all'
INPUT_DIR = "/content/fas2rdna" # please do not change
OUTPUT_DIR = "/content/fas2rdna/outputs" # please do not change
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
Project_name = "" # @param {"type":"string","placeholder":"e.g., My experiment"}
%cd {INPUT_DIR}
upload_files = files.upload()  # User uploads one or more TXT files

In [ ]:
import os
from google.colab import files

# @title ##2. Run configuration
# @markdown ### A. Directories
# @markdown ###### Please do not change the locations of input and output
Input_location = "/content/fas2rdna"  # @param {type:"string"}
Output_location = "/content/fas2rdna/outputs"  # @param {type:"string"}

# @markdown ### B. Assemblies
# @markdown ###### Refer to: www.genome.ucsc.edu/cgi-bin/hgGateway for assembly details (FAS2rDNA will detect them automatically)
Genome_assemblies = "hg16, hg17, hg18, hg19, hg38, hs1, mm7, mm8, mm9, mm10, mm39, rn4, rn5, rn6, rn7, danRer7, danRer10, danRer11, dm2, dm3, dm6, ce4, ce6, ce10, ce11, sacCer1, sacCer2, sacCer3" # @param {"type":"string","placeholder":"e.g., hg18, hg19"}

SUPPORTED_ASSEMBLIES = {
    # Human
    "hg16", "hg17", "hg18", "hg19", "hg38", "hs1",

    # Mouse
    "mm7", "mm8", "mm9", "mm10", "mm39",

    # Rat
    "rn4", "rn5", "rn6", "rn7",

    # Zebrafish
    "danRer7", "danRer10", "danRer11",

    # Fruitfly
    "dm2", "dm3", "dm6",

    # C. elegans
    "ce4", "ce6", "ce10", "ce11",

    # S. cerevisiae
    "sacCer1", "sacCer2", "sacCer3"
}

# Parse user input
ASSEMBLIES = [
    asm.strip()
    for asm in Genome_assemblies.split(",")
    if asm.strip()
]

if not ASSEMBLIES:
    raise ValueError("No assemblies provided. Please enter at least one assembly.")

# Validate assemblies
invalid = [asm for asm in ASSEMBLIES if asm not in SUPPORTED_ASSEMBLIES]
if invalid:
    raise ValueError(
        f"Invalid assembly name(s): {', '.join(invalid)}\n"
        f"Supported assemblies are: {', '.join(sorted(SUPPORTED_ASSEMBLIES))}"
    )

# @markdown ### C. FASTA Headers
# @markdown ###### FAS2rDNA uses the 'sample_id' row to generate headers
Sample_header = ">sample_id" # @param {"type":"string"}

In [ ]:
# @title ##3. Install dependencies

!pip install pandas pyfaidx tqdm
!apt-get update -qq
!apt-get install -y samtools

import zipfile
import re
import sys
import platform
import os
import pandas as pd
from pyfaidx import Fasta
from tqdm import tqdm
import subprocess
import urllib.request
import gzip
import shutil

# Clone FAS2rDNA-Colab from GitHub
get_fast2rdna = "https://github.com/mahvin92/FAS2rDNA-Colab.git"
save_fast2rdna = "/content/fas2rdna"
!git init
!git remote add origin {get_fast2rdna}
!git config core.sparseCheckout true
!echo "source/*" >> .git/info/sparse-checkout
!git pull origin main
!mv source/* .
!rm -rf source .git
!pip install /content/fas2rdna/install/codeenigma_runtime-1.2.0-py3-none-any.whl


##4. Run FAS2rDNA

In [ ]:
import fas2rdna

def fas2rdnaO(source_path):
    clean_name = re.sub(r'[^\w\s-]', '', Project_name).strip().replace(' ', '_')
    zip_filename = f"{clean_name}_fasta_results.zip" if clean_name else "fasta_outputs.zip"

    if not os.path.exists(source_path):
        print(f"Error: The path {source_path} does not exist.")
        return

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        file_count = 0
        for root, dirs, files_list in os.walk(source_path):
            for file in files_list:
                if file.lower().endswith('.fasta'):
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, source_path)
                    zipf.write(file_path, arcname)
                    file_count += 1

        if file_count == 0:
            print("No .fasta files found. Nothing to download.")
            return

    print(f"Downloading FAS2rDNA results: {file_count} fasta files ...")
    files.download(zip_filename)

fas2rdnaO(OUTPUT_DIR)